## Structured Output

Models can be requested to provide response in a format matching a given schema. 

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

model = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    google_api_key = os.getenv("GOOGLE_API_KEY")
)

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title : str = Field(description="Title of movie")
    year : int = Field(description="Year the movie was released")
    director : str = Field(description="Director of the field")
    rating : float = Field(description="Movies rating out of 10")


In [10]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.2'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x000002A3461C12B0>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_jso

In [8]:
model_with_structure.invoke("Provide details about movie Shutter island")

Movie(title='Shutter Island', year=2010, director='Martin Scorsese', rating=8.2)

### Message output alongside parsed structure

In [11]:
model_with_structure = model.with_structured_output(Movie,include_raw=True)
model_with_structure 

{
  raw: _ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.2'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x000002A3461C12B0>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'res

In [12]:
response = model_with_structure.invoke("Provide details about movie Shutter island")
response

{'raw': AIMessage(content='{"title":"Shutter Island","year":2010,"director":"Martin Scorsese","rating":8.2}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a00592-24ef-7c10-9510-dd8379e8a70d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 132, 'total_tokens': 140, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 107}}),
 'parsed': Movie(title='Shutter Island', year=2010, director='Martin Scorsese', rating=8.2),
 'parsing_error': None}

### Nested Structure

In [14]:
class Actor(BaseModel) : 
    name : str
    role : str

class MovieDetails(BaseModel):
    title : str
    year : int
    cast : list[Actor]
    genres : list[str]
    budget : float | None = Field(None, description="Budget in millions USD")
    box_office_collection : float | None = Field(None, description="In millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Michael Fischer')], genres=['Sci-Fi', 'Action', 'Thriller'], budget=160.0, box_office_collection=836.8)